# Once Class SVM to detect anomaly
- https://www.kaggle.com/code/amarnayak/once-class-svm-to-detect-anomaly

### Load Dataset

In [ ]:
import pandas as pd
import numpy as np
from sklearn import svm


cc =  pd.read_csv("../input/creditcard.csv")

# Data check. 
cc.head()

# I observed an conflict in the name 'class'. 
# Therefore, I have changed the name from class to category
cc= cc.rename(columns={'Class': 'Category'})

# For convinience, divide the dataframe cc based on two labels. 
nor_obs = cc.loc[cc.Category==0]    #Data frame with normal observation
ano_obs = cc.loc[cc.Category==1]    #Data frame with anomalous observation

### Setting up TrainSet and TestSet
- Once class SVM is `trained` with the observations of only one class (`only normal samples`). 
- In this case, the algorithm is trained with `first 200,000 observation of normal transactions (a train set)`. 
- The `remaining observations (normal samples)` are merged with the `anomalous observation` to create `a test set`.

In [ ]:
# create train_feature: the first 200,000 normal observation
train_feature = nor_obs.loc[0:200000, :]
train_feature = train_feature.drop('Category', 1)

# Y_1: the remaining normal observation
# Y_2: the abnormal observation
# Y_test = (Y_1 + Y_2): is used to evaluate the model
# Y_test is used to compare with predicted result
Y_1 = nor_obs.loc[200000:, 'Category']
Y_2 = ano_obs['Category']
Y_test= Y_1.append(Y_2)

# Creatng test observations/features
X_test_1 = nor_obs.loc[200000:, :].drop('Category',1)
X_test_2 = ano_obs.drop('Category',1)
X_test = X_test_1.append(X_test_2)

## Training `Once Class SVM`

In [ ]:
from sklearn import svm

# Setting the hyperparameters for Once Class SVM
oneclass = svm.OneClassSVM(kernel='linear', gamma=0.001, nu=0.95)

# Training the algorithm with the features. 
# This stage is very time consuming processes. 
# In my laptop it took more than an hour to train for 200,000 observations. 
# For rbf, the time taken is even more.
oneclass.fit(train_feature)

###  Test the algorithm on the TestSet

In [ ]:
fraud_pred = oneclass.predict(X_test)

In [ ]:
# Check the number of outliers predicted by the algorithm
unique, counts = np.unique(fraud_pred, return_counts=True)
print (np.asarray((unique, counts)).T)

In [ ]:
#Convert Y-test and fraud_pred to dataframe for ease of operation

# Y_test = (Y_1 + Y_2): is used to evaluate the model
# Y_test is used to compare with predicted result
Y_test= Y_test.to_frame()
Y_test=Y_test.reset_index()

fraud_pred = pd.DataFrame(fraud_pred)
fraud_pred= fraud_pred.rename(columns={0: 'prediction'})

In [ ]:
##Performance check of the model

TP = FN = FP = TN = 0
# Category== 1: anomaly, predict== -1
# Category== 0: normal, predict== 1
for j in range(len(Y_test)):
    if Y_test['Category'][j]== 0 and fraud_pred['prediction'][j] == 1:
        TP = TP+1
    elif Y_test['Category'][j]== 0 and fraud_pred['prediction'][j] == -1:
        FN = FN+1
    elif Y_test['Category'][j]== 1 and fraud_pred['prediction'][j] == 1:
        FP = FP+1
    else:
        TN = TN +1
print (TP,  FN,  FP,  TN)

In [ ]:
# Performance Matrix

accuracy = (TP+TN)/(TP+FN+FP+TN)
print (accuracy)

sensitivity = TP/(TP+FN)
print (sensitivity)

specificity = TN/(TN+FP)
print (specificity)

- Following results were obtained
    - accuracy= 99.9%
    - sensitivity = 100%
    - specificity = 75%

- Once class SVM has shown a very promising performance for this dataset with near 90% detection of anomaly and very few false alarm. This can be a starting point for fine tuning the algorthm to improve the specificity, keeping other things constant. 

- Tuning the hyperparameters are very time consuming process and the Kaggle kernal stops after some time. Therefore, O couldnt run the code. I have just shown my codes in the cell. I am sure this code will run because i have ran it in my Jupyter note book. 

- I have also isolation forest in my previous kernal. Once class SVM seems to outperform isolation forest in this case.